In [8]:
# Latest stable versions — works with Colab's Python 3.10+
!pip install transformers  evaluate datasets seqeval accelerate -q

In [9]:
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)
import evaluate

# Reproducibility
import torch
torch.manual_seed(42)

In [10]:
# WikiANN (pan_x): multilingual NER — we use English here.
# Labels: O, B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC  (IOB2)
raw = load_dataset("wikiann", "en")
print(raw)
print("Example:", raw["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

en/validation-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/test-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/train-00000-of-00001.parquet:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})
Example: {'tokens': ['R.H.', 'Saunders', '(', 'St.', 'Lawrence', 'River', ')', '(', '968', 'MW', ')'], 'ner_tags': [3, 4, 0, 3, 4, 4, 0, 0, 0, 0, 0], 'langs': ['en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en'], 'spans': ['ORG: R.H. Saunders', 'ORG: St. Lawrence River']}


In [11]:
# WikiANN integer → IOB string
label_list = raw["train"].features["ner_tags"].feature.names
print("Labels:", label_list)
# → ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']

# Build lookup dicts for the model
id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in id2label.items()}
num_labels = len(label_list)
print(f"Num labels: {num_labels}")

Labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']
Num labels: 7


In [12]:
MODEL_NAME = "bert-base-cased"   # cased matters for NER (names!)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    """
    BERT splits words into sub-tokens. We only label the FIRST
    sub-token of each word; the rest get -100 (ignored by loss).
    This is standard IOB alignment for token classification.
    """
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,  # input is already word-tokenized
        max_length=128,
    )
    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word_id = None
        aligned = []
        for wid in word_ids:
            if wid is None:          # [CLS] / [SEP]
                aligned.append(-100)
            elif wid != prev_word_id: # first sub-token → real label
                aligned.append(labels[wid])
            else:                      # subsequent sub-tokens → ignore
                aligned.append(-100)
            prev_word_id = wid
        all_labels.append(aligned)
    tokenized["labels"] = all_labels
    return tokenized

tokenized_datasets = raw.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw["train"].column_names,
)
print(tokenized_datasets)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 20000
    })
})


In [13]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)
print(f"Parameters: {model.num_parameters():,}")

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Parameters: 107,725,063


In [14]:
# DataCollatorForTokenClassification pads both input_ids AND labels
# to the same length within each batch (dynamic padding = faster training)
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,   # padding labels are ignored by loss
)

In [15]:
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    # Strip padding (-100) and convert int → string labels
    true_labels, true_preds = [], []
    for pred_row, label_row in zip(predictions, labels):
        row_true, row_pred = [], []
        for p, l in zip(pred_row, label_row):
            if l != -100:
                row_true.append(id2label[l])
                row_pred.append(id2label[p])
        true_labels.append(row_true)
        true_preds.append(row_pred)
    result = seqeval.compute(
        predictions=true_preds,
        references=true_labels,
    )
    return {
        "precision": result["overall_precision"],
        "recall":    result["overall_recall"],
        "f1":        result["overall_f1"],
        "accuracy":  result["overall_accuracy"],
    }

In [16]:
args = TrainingArguments(
    output_dir="bert-ner-wikiann",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    push_to_hub=False,
    report_to="none",       # set "wandb" if you use W&B
    fp16=True,              # free speed on Colab T4/V100
)

In [17]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
print("✓ Training done")

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.587000,0.251594,0.783248,0.819309,0.800872,0.922965
2,0.228700,0.235095,0.804968,0.833192,0.818837,0.928582
3,0.181800,0.239913,0.821896,0.844950,0.833263,0.932136


✓ Training done


In [18]:
results = trainer.evaluate(tokenized_datasets["test"])
print(f"Test F1       : {results['eval_f1']:.4f}")
print(f"Test Precision: {results['eval_precision']:.4f}")
print(f"Test Recall   : {results['eval_recall']:.4f}")

Test F1       : 0.8325
Test Precision: 0.8190
Test Recall   : 0.8465


In [19]:
from transformers import pipeline

ner_pipe = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",   # merges B/I tokens into spans
    device=0 if torch.cuda.is_available() else -1,
)

text = "Elon Musk founded SpaceX in Hawthorne, California."
entities = ner_pipe(text)
for ent in entities:
    print(f"  [{ent['entity_group']:5s}] {ent['word']:<20s} score={ent['score']:.3f}")

  [PER  ] Elon Musk            score=0.891
  [ORG  ] SpaceX               score=0.961
  [LOC  ] Hawthorne, California score=0.963


In [20]:
# Save locally in Colab (or push to Hub)
trainer.save_model("bert-ner-wikiann-final")
tokenizer.save_pretrained("bert-ner-wikiann-final")
print("Saved to bert-ner-wikiann-final/")

# Optional: push to HuggingFace Hub
# trainer.push_to_hub("your-username/bert-ner-wikiann")

Saved to bert-ner-wikiann-final/
